In [ ]:
import json
import math
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

plt.style.use("default")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
BASE_DIR = Path("homeworks") / "HW12"
DATA_PATH = BASE_DIR / "S12-hw-dataset.csv"
ARTIFACTS_DIR = BASE_DIR / "artifacts"
FIGURES_DIR = ARTIFACTS_DIR / "figures"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Не найден файл датасета: {DATA_PATH}. "
        "Положи S12-hw-dataset.csv в homeworks/HW12/ и перезапусти ноутбук."
    )

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
required_cols = {"date", "target"}
missing_required = required_cols - set(df.columns)
if missing_required:
    raise ValueError(f"В датасете не хватает обязательных колонок: {missing_required}")

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values("date").reset_index(drop=True)

print("Размер датасета:", df.shape)
print("Диапазон дат:", df["date"].min(), "—", df["date"].max())
print("\nПропуски по колонкам:")
display(df.isna().sum().to_frame("missing_count"))

display(df.head())
display(df.tail())

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df["date"], df["target"], linewidth=1)
ax.set_title("Базовый график временного ряда")
ax.set_xlabel("date")
ax.set_ylabel("target")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "raw_series.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
n = len(df)
train_ratio, val_ratio, test_ratio = 0.70, 0.15, 0.15
train_end = int(n * train_ratio)
val_end = int(n * (train_ratio + val_ratio))

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

train_end_date = train_df["date"].iloc[-1]
val_end_date = val_df["date"].iloc[-1]
test_start_date = test_df["date"].iloc[0]

split_summary = {
    "train_rows": int(len(train_df)),
    "val_rows": int(len(val_df)),
    "test_rows": int(len(test_df)),
    "train_end_date": str(train_end_date),
    "val_end_date": str(val_end_date),
    "test_start_date": str(test_start_date),
}

print(json.dumps(split_summary, ensure_ascii=False, indent=2))

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df["date"], df["target"], linewidth=1, color="black", alpha=0.7)

ax.axvspan(df["date"].iloc[0], train_end_date, alpha=0.12)
ax.axvspan(train_end_date, val_end_date, alpha=0.12)
ax.axvspan(val_end_date, df["date"].iloc[-1], alpha=0.12)

ax.axvline(train_end_date, linestyle="--")
ax.axvline(val_end_date, linestyle="--")

ax.set_title("Temporal split: train / validation / test")
ax.set_xlabel("date")
ax.set_ylabel("target")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "series_split.png", dpi=160, bbox_inches="tight")
plt.show()

print("Почему random split некорректен для временного ряда:")
print("- он перемешивает прошлое и будущее, нарушая причинность;")
print("- лаговые признаки и rolling-статистики начинают использовать информацию из будущего;")
print("- метрики становятся завышенными и не отражают реальный сценарий прогноза.")

In [ ]:
def mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def evaluate(y_true, y_pred):
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": rmse(y_true, y_pred),
        "mape": mape(y_true, y_pred),
    }

def to_jsonable(obj):
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if isinstance(obj, (pd.Timestamp,)):
        return str(obj)
    return obj

In [ ]:
# B1: naive-last
df_b1 = df.copy()
df_b1["pred"] = df_b1["target"].shift(1)

b1_val_mask = (df_b1["date"] > train_end_date) & (df_b1["date"] <= val_end_date)
b1_test_mask = df_b1["date"] > val_end_date

b1_val = evaluate(df_b1.loc[b1_val_mask, "target"], df_b1.loc[b1_val_mask, "pred"])
b1_test = evaluate(df_b1.loc[b1_test_mask, "target"], df_b1.loc[b1_test_mask, "pred"])

b1_row = {
    "experiment_id": "B1",
    "task": "forecasting",
    "dataset": DATA_PATH.name,
    "seed": SEED,
    "split_summary": json.dumps(split_summary, ensure_ascii=False),
    "window_size": "",
    "horizon": 1,
    "model_summary": "naive-last",
    "features_summary": "last observed target",
    "scaler": "",
    "optimizer": "",
    "lr": "",
    "epochs_trained": 0,
    "best_val_mae": b1_val["mae"],
    "best_val_rmse": b1_val["rmse"],
    "best_val_mape": b1_val["mape"],
    "test_mae": b1_test["mae"],
    "test_rmse": b1_test["rmse"],
    "test_mape": b1_test["mape"],
    "notes": "Baseline: prediction equals previous observed value.",
}

b1_row

In [ ]:
# B2: moving average
ma_window = 7
df_b2 = df.copy()
df_b2["pred"] = df_b2["target"].rolling(window=ma_window, min_periods=ma_window).mean().shift(1)

b2_val = evaluate(df_b2.loc[b1_val_mask, "target"], df_b2.loc[b1_val_mask, "pred"])
b2_test = evaluate(df_b2.loc[b1_test_mask, "target"], df_b2.loc[b1_test_mask, "pred"])

b2_row = {
    "experiment_id": "B2",
    "task": "forecasting",
    "dataset": DATA_PATH.name,
    "seed": SEED,
    "split_summary": json.dumps(split_summary, ensure_ascii=False),
    "window_size": ma_window,
    "horizon": 1,
    "model_summary": f"moving-average(window={ma_window})",
    "features_summary": "rolling mean of past target values",
    "scaler": "",
    "optimizer": "",
    "lr": "",
    "epochs_trained": 0,
    "best_val_mae": b2_val["mae"],
    "best_val_rmse": b2_val["rmse"],
    "best_val_mape": b2_val["mape"],
    "test_mae": b2_test["mae"],
    "test_rmse": b2_test["rmse"],
    "test_mape": b2_test["mape"],
    "notes": "Baseline: rolling average over past observations only.",
}

b2_row

In [ ]:
def add_time_features(frame):
    out = frame.copy()
    out["dayofweek"] = out["date"].dt.dayofweek
    out["month"] = out["date"].dt.month
    out["dayofmonth"] = out["date"].dt.day
    out["dayofyear"] = out["date"].dt.dayofyear
    out["is_weekend"] = out["dayofweek"].isin([5, 6]).astype(int)
    return out

def add_lag_rolling_features(frame):
    out = frame.copy()
    out["lag_1"] = out["target"].shift(1)
    out["lag_7"] = out["target"].shift(7)
    out["lag_14"] = out["target"].shift(14)
    out["rolling_mean_7"] = out["target"].shift(1).rolling(window=7, min_periods=7).mean()
    out["rolling_std_7"] = out["target"].shift(1).rolling(window=7, min_periods=7).std()
    return out

df_feat = add_time_features(add_lag_rolling_features(df))

feature_cols = [
    "lag_1",
    "lag_7",
    "lag_14",
    "rolling_mean_7",
    "rolling_std_7",
    "dayofweek",
    "month",
    "dayofmonth",
    "dayofyear",
    "is_weekend",
]

df_feat = df_feat.dropna(subset=feature_cols).reset_index(drop=True)

train_feat = df_feat[df_feat["date"] <= train_end_date].copy()
val_feat = df_feat[(df_feat["date"] > train_end_date) & (df_feat["date"] <= val_end_date)].copy()
test_feat = df_feat[df_feat["date"] > val_end_date].copy()

X_train = train_feat[feature_cols].values
y_train = train_feat["target"].values
X_val = val_feat[feature_cols].values
y_val = val_feat["target"].values
X_test = test_feat[feature_cols].values
y_test = test_feat["target"].values

ridge_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0, random_state=SEED)),
])

ridge_pipe.fit(X_train, y_train)

pred_val_b3 = ridge_pipe.predict(X_val)
pred_test_b3 = ridge_pipe.predict(X_test)

b3_val = evaluate(y_val, pred_val_b3)
b3_test = evaluate(y_test, pred_test_b3)

b3_row = {
    "experiment_id": "B3",
    "task": "forecasting",
    "dataset": DATA_PATH.name,
    "seed": SEED,
    "split_summary": json.dumps(split_summary, ensure_ascii=False),
    "window_size": "",
    "horizon": 1,
    "model_summary": "Ridge(alpha=1.0)",
    "features_summary": ", ".join(feature_cols),
    "scaler": "StandardScaler(on train only)",
    "optimizer": "",
    "lr": "",
    "epochs_trained": 0,
    "best_val_mae": b3_val["mae"],
    "best_val_rmse": b3_val["rmse"],
    "best_val_mape": b3_val["mape"],
    "test_mae": b3_test["mae"],
    "test_rmse": b3_test["rmse"],
    "test_mape": b3_test["mape"],
    "notes": "Linear baseline on lag, rolling and calendar features.",
}

b3_row

### Почему в признаках нет утечки
Все лаговые и rolling-признаки строятся через `shift(1)`, то есть для строки в момент времени `t` используются только значения `<= t-1`.

### Почему масштабирование делается только по train
`StandardScaler` обучается на train и затем применяется к validation/test, чтобы валидационная и тестовая части не влияли на параметры преобразования.

In [ ]:
# Сравнение baseline-моделей по validation
runs_baseline = pd.DataFrame([b1_row, b2_row, b3_row])
display(runs_baseline[["experiment_id", "best_val_mae", "best_val_rmse", "best_val_mape", "test_mae", "test_rmse", "test_mape"]])

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(runs_baseline["experiment_id"], runs_baseline["best_val_mae"])
ax.set_title("Сравнение baseline-моделей по validation MAE")
ax.set_xlabel("experiment_id")
ax.set_ylabel("validation MAE")
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "baselines_compare.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, series, target_indices, window_size):
        self.series = np.asarray(series, dtype=np.float32)
        self.target_indices = np.asarray(target_indices, dtype=np.int64)
        self.window_size = window_size

    def __len__(self):
        return len(self.target_indices)

    def __getitem__(self, idx):
        target_idx = self.target_indices[idx]
        x = self.series[target_idx - self.window_size:target_idx]
        y = self.series[target_idx]
        return torch.tensor(x[:, None], dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

def build_window_indices(n_total, window_size, train_end_pos, val_end_pos):
    target_indices = np.arange(window_size, n_total)
    train_mask = target_indices < train_end_pos
    val_mask = (target_indices >= train_end_pos) & (target_indices < val_end_pos)
    test_mask = target_indices >= val_end_pos
    return target_indices[train_mask], target_indices[val_mask], target_indices[test_mask]

class GRURegressor(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1, dropout=0.0):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        last = out[:, -1, :]
        return self.head(last).squeeze(-1)

def inverse_transform_1d(scaler, arr):
    arr = np.asarray(arr).reshape(-1, 1)
    return scaler.inverse_transform(arr).ravel()

def predict_loader(model, loader, target_scaler, device):
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            pred = model(xb).cpu().numpy()
            preds.append(pred)
            trues.append(yb.numpy())
    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    preds_inv = inverse_transform_1d(target_scaler, preds)
    trues_inv = inverse_transform_1d(target_scaler, trues)
    return trues_inv, preds_inv

def train_gru(model, train_loader, val_loader, target_scaler, device, lr=1e-3, epochs=50, patience=8):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    best_state = None
    best_val_mae = float("inf")
    best_epoch = -1
    patience_left = patience

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))

        y_val_true, y_val_pred = predict_loader(model, val_loader, target_scaler, device)
        val_metrics = evaluate(y_val_true, y_val_pred)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_mae": val_metrics["mae"],
            "val_rmse": val_metrics["rmse"],
            "val_mape": val_metrics["mape"],
        })

        if val_metrics["mae"] < best_val_mae:
            best_val_mae = val_metrics["mae"]
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history), best_epoch, best_val_mae, best_state

In [ ]:
window_size = 28
hidden_size = 64
batch_size = 64
lr = 1e-3
epochs = 50
patience = 8

target_scaler = StandardScaler()
target_scaler.fit(train_df[["target"]])

target_scaled = target_scaler.transform(df[["target"]]).ravel()
n_total = len(target_scaled)

train_target_end = train_end
val_target_end = val_end

train_idx, val_idx, test_idx = build_window_indices(
    n_total=n_total,
    window_size=window_size,
    train_end_pos=train_target_end,
    val_end_pos=val_target_end,
)

train_ds = WindowDataset(target_scaled, train_idx, window_size)
val_ds = WindowDataset(target_scaled, val_idx, window_size)
test_ds = WindowDataset(target_scaled, test_idx, window_size)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, drop_last=False)

gru = GRURegressor(input_size=1, hidden_size=hidden_size, num_layers=1, dropout=0.0).to(device)

gru, gru_history, best_epoch, best_val_mae, best_state = train_gru(
    model=gru,
    train_loader=train_loader,
    val_loader=val_loader,
    target_scaler=target_scaler,
    device=device,
    lr=lr,
    epochs=epochs,
    patience=patience,
)

y_val_true_gru, y_val_pred_gru = predict_loader(gru, val_loader, target_scaler, device)
y_test_true_gru, y_test_pred_gru = predict_loader(gru, test_loader, target_scaler, device)

gru_val = evaluate(y_val_true_gru, y_val_pred_gru)
gru_test = evaluate(y_test_true_gru, y_test_pred_gru)

best_gru_path = ARTIFACTS_DIR / "best_gru.pt"
torch.save(gru.state_dict(), best_gru_path)

best_gru_config = {
    "experiment_id": "R1",
    "task": "forecasting",
    "dataset": DATA_PATH.name,
    "seed": SEED,
    "window_size": window_size,
    "horizon": 1,
    "input_size": 1,
    "hidden_size": hidden_size,
    "num_layers": 1,
    "dropout": 0.0,
    "batch_size": batch_size,
    "lr": lr,
    "epochs_trained": int(gru_history.shape[0]),
    "patience": patience,
    "scaler": "StandardScaler(on train target only)",
    "best_epoch": int(best_epoch),
    "best_val_mae": float(best_val_mae),
}
(best_gru_path.parent / "best_gru_config.json").write_text(
    json.dumps(best_gru_config, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

gru_row = {
    "experiment_id": "R1",
    "task": "forecasting",
    "dataset": DATA_PATH.name,
    "seed": SEED,
    "split_summary": json.dumps(split_summary, ensure_ascii=False),
    "window_size": window_size,
    "horizon": 1,
    "model_summary": f"GRU(hidden_size={hidden_size}, num_layers=1)",
    "features_summary": "windowed target sequence",
    "scaler": "StandardScaler(on train target only)",
    "optimizer": "Adam",
    "lr": lr,
    "epochs_trained": int(gru_history.shape[0]),
    "best_val_mae": gru_val["mae"],
    "best_val_rmse": gru_val["rmse"],
    "best_val_mape": gru_val["mape"],
    "test_mae": gru_test["mae"],
    "test_rmse": gru_test["rmse"],
    "test_mape": gru_test["mape"],
    "notes": f"Best epoch={best_epoch}; early stopping on validation MAE.",
}

gru_row

In [ ]:
# Кривые обучения GRU
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(gru_history["epoch"], gru_history["train_loss"], label="train loss")
ax1.set_xlabel("epoch")
ax1.set_ylabel("train loss")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(gru_history["epoch"], gru_history["val_mae"], label="val MAE")
ax2.set_ylabel("validation MAE")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

ax1.set_title("GRU learning curves")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "gru_learning_curves.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# Финальный прогноз на test для лучшей модели
test_dates = df.iloc[test_idx]["date"].values

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(test_dates, y_test_true_gru, label="actual", linewidth=2)
ax.plot(test_dates, y_test_pred_gru, label="forecast", linewidth=2)
ax.set_title("Best GRU forecast on test")
ax.set_xlabel("date")
ax.set_ylabel("target")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "best_forecast_test.png", dpi=160, bbox_inches="tight")
plt.show()

residuals = y_test_true_gru - y_test_pred_gru
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(test_dates, residuals, linewidth=1)
ax.axhline(0, linestyle="--")
ax.set_title("Residuals of best GRU on test")
ax.set_xlabel("date")
ax.set_ylabel("error")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "residuals_best.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
runs = pd.DataFrame([b1_row, b2_row, b3_row, gru_row])

# Удобный порядок колонок
preferred_cols = [
    "experiment_id", "task", "dataset", "seed", "split_summary", "window_size", "horizon",
    "model_summary", "features_summary", "scaler", "optimizer", "lr", "epochs_trained",
    "best_val_mae", "best_val_rmse", "best_val_mape", "test_mae", "test_rmse", "test_mape", "notes"
]
runs = runs[[c for c in preferred_cols if c in runs.columns]]

runs_path = ARTIFACTS_DIR / "runs.csv"
runs.to_csv(runs_path, index=False, encoding="utf-8")

display(runs.sort_values("experiment_id"))

print(f"Сохранено: {runs_path}")
print(f"Сохранено: {best_gru_path}")
print(f"Сохранено: {ARTIFACTS_DIR / 'best_gru_config.json'}")
print(f"Графики: {FIGURES_DIR}")